# 01 — Preparação dos dados curriculares

Este notebook reúne o fluxo utilizado nos testes suplementares com currículos Lattes:

1. leitura dos XMLs Lattes;
2. anonimização e remoção de campos pessoais;
3. seleção dos autores pertencentes ao conjunto experimental;
4. estruturação e compactação das seções curriculares;
5. aplicação dos limites utilizados nos experimentos;
6. geração de `curriculos_estruturados_padrao.json`;
7. geração de `perfis_documento_curriculo.json` para o cálculo de Coverage;
8. validações de estrutura, alinhamento com os qrels e auditoria das saídas.


In [ ]:
from pathlib import Path
import os

current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "data").exists() or (p / "src").exists()), current)

# -------------------------------------------------------------------------
# ENTRADAS
# -------------------------------------------------------------------------
# Pasta contendo os XMLs Lattes.
PASTA_XML = PROJECT_ROOT / "data" / "curriculos" / "lattes-prof-qrels-xml"

# Qrels filtrados que definem os autores pertencentes ao conjunto experimental.
ARQUIVO_AUTORES = PROJECT_ROOT / "data" / "processed" / "ground_truth" / "LExR-prof-qrels_filtrado"

# -------------------------------------------------------------------------
# SAÍDAS
# -------------------------------------------------------------------------
CURRICULO_DIR = PROJECT_ROOT / "data" / "curriculos"
CURRICULO_DIR.mkdir(parents=True, exist_ok=True)

ARQUIVO_SAIDA = CURRICULO_DIR / "curriculos_estruturados_padrao.json"
CAMINHO_SAIDA_COVERAGE = CURRICULO_DIR / "perfis_documento_curriculo.json"
CAMINHO_RELATORIO = CURRICULO_DIR / "processing_report_perfis_documento_curriculo.json"

print("Pasta XML:", PASTA_XML)
print("Qrels:", ARQUIVO_AUTORES)
print("JSON estruturado:", ARQUIVO_SAIDA)
print("Coverage:", CAMINHO_SAIDA_COVERAGE)

if not PASTA_XML.is_dir():
    raise FileNotFoundError(f"Pasta XML não encontrada: {PASTA_XML}")

if not ARQUIVO_AUTORES.is_file():
    raise FileNotFoundError(f"Arquivo de autores não encontrado: {ARQUIVO_AUTORES}")

In [ ]:
# =============================================================================
# CÉLULA 2 — Imports
# =============================================================================

import json
import re
import xml.etree.ElementTree as ET

from collections import defaultdict, Counter
from typing import Dict, List, Optional, Set
from tqdm.auto import tqdm


In [ ]:
# =============================================================================
# CÉLULA 3 — Configuração da extração dos XMLs
# =============================================================================

# O resumo fica desligado porque é texto livre e pode conter identificação nominal.
INCLUIR_RESUMO_CV = False

# Tags inteiras removidas na leitura do XML.
TAGS_REMOVER_INTEIRAS = {
    "ENDERECO",
    "ENDERECO-PROFISSIONAL",
    "AUTORES",
    "PARTICIPANTE-BANCA",
    "INFORMACOES-ADICIONAIS",
    "INFORMACOES-ADICIONAIS-INSTITUICOES",
    "INFORMACOES-ADICIONAIS-CURSOS",
}

# Campos pessoais/nominais removidos.
CAMPOS_PESSOAIS = {
    "NOME-COMPLETO",
    "NOME-COMPLETO-DO-AUTOR",
    "NOME-PARA-CITACAO",
    "NOME-CITACAO",
    "NOME-DO-CANDIDATO",
    "NOME-COMPLETO-DO-ORIENTADOR",
    "NOME-DO-ORIENTADOR",
    "NOME-COMPLETO-DO-PARTICIPANTE-DA-BANCA",
    "NOME-DO-PARTICIPANTE",
    "NOME-COMPLETO-DO-CO-AUTOR",
    "CPF",
    "NUMERO-DO-CPF",
    "ORCID-ID",
    "ID-ORCID",
    "E-MAIL",
    "EMAIL",
    "TELEFONE",
    "CELULAR",
    "FAX",
    "DATA-NASCIMENTO",
    "DATA-DE-NASCIMENTO",
    "CIDADE-NASCIMENTO",
    "UF-NASCIMENTO",
    "PAIS-DE-NASCIMENTO",
    "NACIONALIDADE",
    "SEXO",
    "NUMERO-IDENTIFICADOR",
}

# Metadados institucionais/técnicos retirados pelo extrator original.
CAMPOS_SEM_VALOR_EXPERTISE = {
    "NOME-INSTITUICAO",
    "NOME-DA-INSTITUICAO",
    "NOME-AGENCIA",
    "SEQUENCIA-FORMACAO",
    "SEQUENCIA-ATIVIDADE",
    "SEQUENCIA-IMPORTANCIA",
    "SEQUENCIA-HISTORICO",
    "SEQUENCIA-FUNCAO-ATIVIDADE",
    "SEQUENCIA-ESPECIFICACAO",
    "NIVEL",
    "FLAG-BOLSA",
    "FLAG-DEDICACAO-EXCLUSIVA",
    "FLAG-VINCULO-EMPREGATICIO",
    "DOI",
    "ISBN",
    "ISSN",
    "VOLUME",
    "FASCICULO",
    "SERIE",
    "PAGINA-INICIAL",
    "PAGINA-FINAL",
}


In [ ]:
# =============================================================================
# CÉLULA 4 — Funções de leitura, anonimização e conversão dos XMLs
# =============================================================================

def remover_atributo(nome):
    nome = nome.upper().strip()
    if nome in CAMPOS_PESSOAIS or nome in CAMPOS_SEM_VALOR_EXPERTISE:
        return True
    if nome.startswith('CODIGO-'):
        return True
    if nome.startswith('HOME-PAGE') or 'URL' in nome:
        return True
    if nome.startswith('SEQUENCIA-'):
        return True
    if ('AUTOR' in nome or 'ORIENTADOR' in nome or 'CANDIDATO' in nome or 'PARTICIPANTE' in nome) and 'NOME' in nome:
        return True
    if 'INSTITUICAO' in nome and nome.startswith('NOME'):
        return True
    return False


def remover_namespace(tag):
    return tag.split('}', 1)[1] if '}' in tag else tag


def normalizar_id(valor):
    valor = str(valor).strip().replace('.xml', '')
    valor = re.sub(r'^ID_', '', valor, flags=re.IGNORECASE)
    numeros = re.findall(r'\d+', valor)
    return numeros[0] if numeros else valor


def limpar_valor(valor):
    if valor is None:
        return None
    valor = re.sub(r'\s+', ' ', str(valor).strip())
    return valor if valor else None


def carregar_autores(caminho):
    ids = set()
    extensao = os.path.splitext(caminho)[1].lower()

    if extensao == '.json':
        with open(caminho, 'r', encoding='utf-8') as f:
            dados = json.load(f)
        if isinstance(dados, dict):
            for chave in dados.keys():
                ids.add(normalizar_id(chave))
        elif isinstance(dados, list):
            for item in dados:
                if isinstance(item, str):
                    ids.add(normalizar_id(item))
                elif isinstance(item, dict):
                    for campo in ['Autor', 'autor', 'ID', 'id', 'author', 'author_id']:
                        if campo in item:
                            ids.add(normalizar_id(item[campo]))
                            break
    else:
        with open(caminho, 'r', encoding='utf-8', errors='ignore') as f:
            for linha in f:
                encontrados = re.findall(r'(?:ID_)?(\d{8,})', linha.strip())
                if encontrados:
                    ids.add(normalizar_id(encontrados[0]))
    return ids


def elemento_para_dict(elemento, ignorar_tags=None):
    ignorar_tags = set() if ignorar_tags is None else set(ignorar_tags)
    tag = remover_namespace(elemento.tag)

    if tag in TAGS_REMOVER_INTEIRAS or tag in ignorar_tags:
        return None

    resultado = {}
    atributos = {}

    for nome, valor in elemento.attrib.items():
        if remover_atributo(nome):
            continue
        valor = limpar_valor(valor)
        if valor is not None:
            atributos[nome] = valor

    if atributos:
        resultado['atributos'] = atributos

    texto = limpar_valor(elemento.text)
    if texto:
        resultado['texto'] = texto

    filhos = defaultdict(list)
    for filho in elemento:
        tag_filho = remover_namespace(filho.tag)
        if tag_filho in TAGS_REMOVER_INTEIRAS or tag_filho in ignorar_tags:
            continue
        conteudo = elemento_para_dict(filho, ignorar_tags=ignorar_tags)
        if conteudo:
            filhos[tag_filho].append(conteudo)

    for tag_filho, conteudos in filhos.items():
        resultado[tag_filho] = conteudos[0] if len(conteudos) == 1 else conteudos

    return resultado or None


def extrair_curriculo(caminho_xml, id_autor):
    try:
        tree = ET.parse(caminho_xml)
        root = tree.getroot()
    except Exception as erro:
        print(f'Erro ao ler {caminho_xml}: {erro}')
        return None

    curriculo = {
        'id_autor': f'ID_{id_autor}',
        'formacao_academica': [],
        'areas_de_atuacao': [],
        'atuacao_profissional': [],
        'linhas_de_pesquisa': [],
        'ensino': [],
        'producao_bibliografica': [],
        'producao_tecnica': [],
        'orientacoes_concluidas': [],
        'demais_trabalhos': [],
        'participacao_em_bancas': []
    }

    if INCLUIR_RESUMO_CV:
        curriculo['resumo_cv'] = []

    for elemento in root.iter():
        tag = remover_namespace(elemento.tag)

        if tag == 'RESUMO-CV' and INCLUIR_RESUMO_CV:
            conteudo = elemento_para_dict(elemento)
            if conteudo:
                curriculo['resumo_cv'].append(conteudo)

        elif tag == 'FORMACAO-ACADEMICA-TITULACAO':
            for filho in elemento:
                conteudo = elemento_para_dict(filho)
                if conteudo:
                    curriculo['formacao_academica'].append({'tipo': remover_namespace(filho.tag), 'dados': conteudo})

        elif tag == 'AREAS-DE-ATUACAO':
            for filho in elemento:
                conteudo = elemento_para_dict(filho)
                if conteudo:
                    curriculo['areas_de_atuacao'].append(conteudo)

        elif tag == 'ATUACOES-PROFISSIONAIS':
            for atuacao in elemento:
                if remover_namespace(atuacao.tag) != 'ATUACAO-PROFISSIONAL':
                    continue
                conteudo = elemento_para_dict(atuacao, ignorar_tags={'LINHA-DE-PESQUISA', 'ATIVIDADES-DE-ENSINO'})
                if conteudo:
                    curriculo['atuacao_profissional'].append(conteudo)

        elif tag == 'LINHA-DE-PESQUISA':
            conteudo = elemento_para_dict(elemento)
            if conteudo:
                curriculo['linhas_de_pesquisa'].append(conteudo)

        elif tag == 'ATIVIDADES-DE-ENSINO':
            for ensino in elemento:
                conteudo = elemento_para_dict(ensino)
                if conteudo:
                    curriculo['ensino'].append(conteudo)

        elif tag == 'PRODUCAO-BIBLIOGRAFICA':
            for filho in elemento:
                conteudo = elemento_para_dict(filho)
                if conteudo:
                    curriculo['producao_bibliografica'].append({'tipo': remover_namespace(filho.tag), 'dados': conteudo})

        elif tag == 'PRODUCAO-TECNICA':
            for filho in elemento:
                conteudo = elemento_para_dict(filho)
                if conteudo:
                    curriculo['producao_tecnica'].append({'tipo': remover_namespace(filho.tag), 'dados': conteudo})

        elif tag == 'ORIENTACOES-CONCLUIDAS':
            for filho in elemento:
                conteudo = elemento_para_dict(filho)
                if conteudo:
                    curriculo['orientacoes_concluidas'].append({'tipo': remover_namespace(filho.tag), 'dados': conteudo})

        elif tag == 'DEMAIS-TRABALHOS':
            conteudo = elemento_para_dict(elemento)
            if conteudo:
                curriculo['demais_trabalhos'].append(conteudo)

        elif tag.startswith('PARTICIPACAO-EM-BANCA-DE-') or tag.startswith('BANCA-JULGADORA-PARA-'):
            conteudo = elemento_para_dict(elemento)
            if conteudo:
                curriculo['participacao_em_bancas'].append({'tipo': tag, 'dados': conteudo})

    return curriculo

In [ ]:
# =============================================================================
# CÉLULA 5 — Carregar os autores do conjunto experimental
# =============================================================================

autores_validos = carregar_autores(ARQUIVO_AUTORES)

print("Autores encontrados no arquivo de referência:", len(autores_validos))

if not autores_validos:
    raise RuntimeError("Nenhum autor foi encontrado em ARQUIVO_AUTORES.")


In [ ]:
# =============================================================================
# CÉLULA 6 — Configuração da estrutura FINAL usada pelos modelos
# =============================================================================

SECOES_USADAS = [
    "formacao_academica",
    "atuacao_profissional",
    "areas_de_atuacao",
    "linhas_de_pesquisa",
    "ensino",
    "producao_bibliografica",
    "producao_tecnica",
    "orientacoes_concluidas",
    "demais_trabalhos",
]

ALIASES_SECOES_CURRICULO = {
    "linhas_de_pesquisa": [
        "linhas_de_pesquisa",
        "linha_de_pesquisa",
    ],
    "orientacoes_concluidas": [
        "orientacoes_concluidas",
        "orientacoes",
    ],
}

# Limites efetivamente usados no código testado.
LIMITES_REGISTROS_CURRICULO = {
    "producao_bibliografica": 30,
    "producao_tecnica": 20,
    "orientacoes_concluidas": 20,
    "demais_trabalhos": 20,
}

print("Seções finais:", SECOES_USADAS)
print("Limites:", LIMITES_REGISTROS_CURRICULO)


In [ ]:
# =============================================================================
# CÉLULA 7 — Limpeza textual usada na estruturação
# =============================================================================

RE_HTML = re.compile(r"<[^>]+>")
RE_ENTIDADE_HTML = re.compile(r"&[a-z]+;|&#\d+;")
RE_CARACTERES_RUIM = re.compile(
    r"[^\w\s.,;:!?()\-'\"áéíóúâêîôûãõàèìòùçÁÉÍÓÚÂÊÎÔÛÃÕÀÈÌÒÙÇñÑüÜ]",
    flags=re.UNICODE,
)
RE_ESPACOS = re.compile(r"\s+")


In [ ]:
# =============================================================================
# CÉLULA 8 — Funções de estruturação, compactação e aplicação dos limites
# =============================================================================

def limpar_texto_basico(texto: str) -> str:
    """
    Limpeza leve, preservando case/acento:
      - tira HTML e entidades
      - tira caracteres não-alfanuméricos exóticos (mantém pontuação básica)
      - colapsa espaços/tabs/quebras múltiplas em um único espaço
    NÃO faz lowercase, NÃO remove acentos, NÃO remove duplicatas.
    """
    if not texto:
        return ""
    texto = RE_HTML.sub(" ", texto)
    texto = RE_ENTIDADE_HTML.sub(" ", texto)
    texto = RE_CARACTERES_RUIM.sub(" ", texto)
    texto = RE_ESPACOS.sub(" ", texto).strip()
    return texto


def normalizar_id_autor_curriculo(autor: str) -> str:
    """Converte ID_123... para 123..., preservando IDs já sem prefixo."""
    autor = str(autor).strip()
    return autor[3:] if autor.startswith("ID_") else autor


def limpar_estrutura_curriculo(obj):
    """
    Limpa recursivamente strings do currículo com limpar_texto_basico e remove
    campos vazios, sem alterar a estrutura semântica do JSON.
    """
    if isinstance(obj, str):
        return limpar_texto_basico(obj)

    if isinstance(obj, list):
        itens = [limpar_estrutura_curriculo(x) for x in obj]
        return [x for x in itens if x not in (None, "", [], {})]

    if isinstance(obj, dict):
        saida = {}
        for chave, valor in obj.items():
            valor_limpo = limpar_estrutura_curriculo(valor)
            if valor_limpo not in (None, "", [], {}):
                saida[chave] = valor_limpo
        return saida

    return obj


def obter_secao_curriculo(curriculo: dict, secao: str):
    """Busca uma seção, aceitando aliases definidos em ALIASES_SECOES_CURRICULO."""
    candidatos = ALIASES_SECOES_CURRICULO.get(secao, [secao])
    for chave in candidatos:
        if chave in curriculo and curriculo[chave] not in (None, "", [], {}):
            return curriculo[chave]
    return None


def extrair_ano_registro(obj) -> int:
    """
    Procura recursivamente valores associados a chaves contendo 'ANO' e
    devolve o maior ano de 4 dígitos encontrado. Se não houver ano válido,
    devolve -1. A função suporta as diferentes nomenclaturas do Lattes
    (ANO, ANO-DO-TRABALHO, ANO-DE-REALIZACAO, etc.).
    """
    anos = []

    def visitar(valor):
        if isinstance(valor, dict):
            for chave, subvalor in valor.items():
                chave_upper = str(chave).upper()
                if "ANO" in chave_upper and isinstance(subvalor, (str, int, float)):
                    m = re.search(r"(?:19|20)\d{2}", str(subvalor))
                    if m:
                        anos.append(int(m.group(0)))
                if isinstance(subvalor, (dict, list)):
                    visitar(subvalor)
        elif isinstance(valor, list):
            for item in valor:
                visitar(item)

    visitar(obj)
    return max(anos) if anos else -1


def limitar_lista_mais_recente(itens, limite: int):
    """
    Ordena uma lista de registros pelo ano (mais recente primeiro) e mantém
    no máximo 'limite'. Registros sem ano ficam por último; empates preservam
    a ordem original graças à estabilidade do sorted().
    """
    if not isinstance(itens, list) or limite is None or limite <= 0:
        return itens

    anotados = [(extrair_ano_registro(item), idx, item) for idx, item in enumerate(itens)]
    anotados.sort(key=lambda x: (x[0] >= 0, x[0]), reverse=True)
    return [item for _, _, item in anotados[:limite]]


def _encontrar_registros_bibliograficos(obj, caminho=()):
    """
    Localiza recursivamente produções bibliográficas individuais. Um registro
    individual é reconhecido por conter alguma chave DADOS-BASICOS-*.
    Retorna pares (caminho, registro), permitindo lidar com diferentes níveis
    de aninhamento do XML/JSON Lattes.
    """
    encontrados = []

    if isinstance(obj, dict):
        if any(str(chave).upper().startswith("DADOS-BASICOS-") for chave in obj.keys()):
            encontrados.append((caminho, obj))
        else:
            for chave, valor in obj.items():
                encontrados.extend(_encontrar_registros_bibliograficos(valor, caminho + (str(chave),)))

    elif isinstance(obj, list):
        for valor in obj:
            encontrados.extend(_encontrar_registros_bibliograficos(valor, caminho))

    return encontrados


def limitar_producao_bibliografica(producao, limite: int = 50):
    """
    Reúne TODAS as produções bibliográficas individuais de todos os blocos
    (artigos, trabalhos em eventos, livros/capítulos etc.), ordena por ano
    decrescente e mantém as 'limite' mais recentes NO TOTAL.

    A saída fica em uma lista plana de registros individuais:
      {"tipo": ..., "categoria": ..., "dados": registro_original}

    Isso faz com que um eventual corte adicional por tokens também preserve
    primeiro as produções mais recentes, registro a registro.
    """
    if not isinstance(producao, list) or limite is None or limite <= 0:
        return producao

    registros = []

    for ordem_bloco, bloco in enumerate(producao):
        if not isinstance(bloco, dict):
            continue

        tipo = bloco.get("tipo", "")
        dados = bloco.get("dados", {})
        encontrados = _encontrar_registros_bibliograficos(dados)

        # Fallback conservador para formatos simples sem DADOS-BASICOS-*.
        if not encontrados and isinstance(dados, dict):
            for chave, valor in dados.items():
                if isinstance(valor, list):
                    encontrados.extend(((str(chave),), item) for item in valor if isinstance(item, dict))
                elif isinstance(valor, dict):
                    encontrados.append(((str(chave),), valor))

        for ordem_item, (caminho, registro) in enumerate(encontrados):
            categoria = " > ".join(caminho) if caminho else str(tipo)
            registros.append({
                "_ano": extrair_ano_registro(registro),
                "_ordem_bloco": ordem_bloco,
                "_ordem_item": ordem_item,
                "tipo": tipo,
                "categoria": categoria,
                "dados": registro,
            })

    # Mais recentes primeiro; em empate, preserva a ordem de origem.
    registros.sort(key=lambda x: (x["_ano"] >= 0, x["_ano"]), reverse=True)
    selecionados = registros[:limite]

    return [
        {
            "tipo": r["tipo"],
            "categoria": r["categoria"],
            "dados": r["dados"],
        }
        for r in selecionados
    ]


def normalizar_nome_campo(nome):
    return re.sub(r"[^A-Z0-9]+", "_", str(nome).upper()).strip("_")


def coletar_campos(obj):
    """
    Percorre recursivamente um registro do Lattes e retorna
    somente pares (nome_do_campo, valor) de valores simples.
    """
    campos = []

    if isinstance(obj, dict):
        for chave, valor in obj.items():
            if isinstance(valor, (dict, list)):
                campos.extend(coletar_campos(valor))
            elif valor not in (None, ""):
                campos.append((normalizar_nome_campo(chave), str(valor).strip()))

    elif isinstance(obj, list):
        for item in obj:
            campos.extend(coletar_campos(item))

    return campos


def valores_campos(campos, nomes):
    nomes = {normalizar_nome_campo(x) for x in nomes}
    resultado = []

    for chave, valor in campos:
        if chave in nomes and valor not in resultado:
            resultado.append(valor)

    return resultado


def primeiro_campo(campos, nomes):
    valores = valores_campos(campos, nomes)
    return valores[0] if valores else None


def valores_campos_contendo(campos, termos):
    termos = [normalizar_nome_campo(x) for x in termos]
    resultado = []

    for chave, valor in campos:
        if any(termo in chave for termo in termos):
            if valor not in resultado:
                resultado.append(valor)

    return resultado


def remover_vazios(d):
    return {
        k: v
        for k, v in d.items()
        if v not in (None, "", [], {})
    }


def compactar_registro_expertise(item, tipo=None):
    campos = coletar_campos(item)

    titulo = primeiro_campo(campos, [
        "TITULO",
        "TITULO-DO-TRABALHO",
        "TITULO-DO-ARTIGO",
        "TITULO-DO-LIVRO",
        "TITULO-DO-CAPITULO-DO-LIVRO",
        "TITULO-DA-PRODUCAO",
        "TITULO-DA-ORIENTACAO",
    ])

    # fallback para outros campos TITULO-...
    if titulo is None:
        for chave, valor in campos:
            if chave.startswith("TITULO_") and "ANAIS" not in chave and "PROCEEDINGS" not in chave:
                titulo = valor
                break

    ano = primeiro_campo(campos, [
        "ANO",
        "ANO-DO-TRABALHO",
        "ANO-DO-ARTIGO",
        "ANO-DO-LIVRO",
        "ANO-DO-CAPITULO",
        "ANO-DA-PRODUCAO",
        "ANO-DE-PUBLICACAO",
        "ANO-DE-REALIZACAO",
        "ANO-DE-CONCLUSAO",
    ])

    natureza = primeiro_campo(campos, [
        "NATUREZA",
        "TIPO",
        "FINALIDADE",
    ])

    palavras_chave = valores_campos_contendo(
        campos,
        ["PALAVRA_CHAVE"]
    )

    areas = valores_campos(campos, [
        "NOME-GRANDE-AREA-DO-CONHECIMENTO",
        "NOME-DA-AREA-DO-CONHECIMENTO",
        "NOME-DA-SUB-AREA-DO-CONHECIMENTO",
        "NOME-DA-ESPECIALIDADE",
    ])

    return remover_vazios({
        "tipo": tipo,
        "ano": ano,
        "titulo": titulo,
        "natureza": natureza,
        "palavras_chave": palavras_chave,
        "areas": areas,
    })


def preparar_producao_bibliografica(conteudo, limite=50):
    """
    Mantém até `limite` PRODUÇÕES BIBLIOGRÁFICAS INDIVIDUAIS mais recentes.

    `limitar_producao_bibliografica()` já devolve uma lista plana em que
    cada item representa UMA publicação:
        {
            "tipo": ...,
            "categoria": ...,
            "dados": registro_completo_da_publicacao
        }

    Portanto, o registro inteiro deve ser compactado de uma só vez.
    Não devemos iterar sobre os subcampos de `dados`, pois isso quebraria uma
    única publicação em vários pseudo-registros (dados básicos, palavras-chave,
    áreas etc.).
    """
    limitado = limitar_producao_bibliografica(
        conteudo,
        limite=limite,
    )

    resultado = []

    for item in limitado:
        if not isinstance(item, dict):
            continue

        tipo = item.get("tipo")
        dados = item.get("dados")

        if not isinstance(dados, dict):
            continue

        compacto = compactar_registro_expertise(
            dados,
            tipo=tipo,
        )

        # Evita manter registros vazios contendo apenas o rótulo "tipo".
        campos_informativos = {
            "ano",
            "titulo",
            "natureza",
            "palavras_chave",
            "areas",
        }

        if compacto and any(
            campo in compacto
            for campo in campos_informativos
        ):
            resultado.append(compacto)

    return resultado[:limite]


def achatar_producao_tecnica(conteudo):
    registros = []

    if not isinstance(conteudo, list):
        return registros

    for bloco in conteudo:

        if not isinstance(bloco, dict):
            continue

        tipo = bloco.get("tipo")
        dados = bloco.get("dados")

        if tipo is not None and isinstance(dados, dict):

            encontrou_lista = False

            for _, valor in dados.items():

                if isinstance(valor, list):
                    encontrou_lista = True

                    for registro in valor:
                        if isinstance(registro, dict):
                            registros.append({
                                "tipo": tipo,
                                "registro": registro
                            })

            # Caso "dados" já represente um único registro
            if not encontrou_lista:
                registros.append({
                    "tipo": tipo,
                    "registro": dados
                })

        else:
            registros.append({
                "tipo": bloco.get("tipo"),
                "registro": bloco
            })

    return registros


def preparar_producao_tecnica(conteudo, limite=50):
    registros = achatar_producao_tecnica(conteudo)

    registros = sorted(
        registros,
        key=lambda x: extrair_ano_registro(x["registro"]),
        reverse=True
    )[:limite]

    resultado = []

    for item in registros:
        compacto = compactar_registro_expertise(
            item["registro"],
            tipo=item["tipo"]
        )

        if compacto:
            resultado.append(compacto)

    return resultado


def preparar_orientacoes(conteudo, limite=50):
    registros = limitar_lista_mais_recente(
        conteudo,
        limite=limite
    )

    resultado = []

    for item in registros:
        tipo = item.get("tipo") if isinstance(item, dict) else None

        compacto = compactar_registro_expertise(
            item,
            tipo=tipo
        )

        if compacto:
            resultado.append(compacto)

    return resultado


def preparar_demais_trabalhos(conteudo, limite=50):
    registros = limitar_lista_mais_recente(
        conteudo,
        limite=limite
    )

    resultado = []

    for item in registros:
        compacto = compactar_registro_expertise(
            item,
            tipo="DEMAIS-TRABALHOS"
        )

        if compacto:
            resultado.append(compacto)

    return resultado


def compactar_atuacao_profissional(conteudo):
    if not isinstance(conteudo, list):
        return conteudo

    resultado = []

    for item in conteudo:
        campos = coletar_campos(item)

        instituicoes = valores_campos(campos, [
            "NOME-INSTITUICAO",
            "NOME-DA-INSTITUICAO",
            "NOME-DA-INSTITUICAO-EMPRESA",
        ])

        vinculos = valores_campos(campos, [
            "TIPO-DE-VINCULO",
            "OUTRO-VINCULO-INFORMADO",
            "ENQUADRAMENTO-FUNCIONAL",
            "OUTRO-ENQUADRAMENTO-FUNCIONAL-INFORMADO",
        ])

        cargos = valores_campos_contendo(
            campos,
            ["CARGO", "FUNCAO"]
        )

        anos_inicio = valores_campos(campos, [
            "ANO-INICIO",
            "ANO-DE-INICIO"
        ])

        anos_fim = valores_campos(campos, [
            "ANO-FIM",
            "ANO-DE-FIM"
        ])

        registro = remover_vazios({
            "instituicao": instituicoes,
            "vinculo_cargo": vinculos + [
                x for x in cargos if x not in vinculos
            ],
            "ano_inicio": anos_inicio,
            "ano_fim": anos_fim,
        })

        if registro:
            resultado.append(registro)

    return resultado


def compactar_formacao_academica(conteudo):
    if not isinstance(conteudo, list):
        return conteudo

    resultado = []

    for item in conteudo:
        campos = coletar_campos(item)

        tipo = item.get("tipo") if isinstance(item, dict) else None

        curso = primeiro_campo(campos, [
            "NOME-DO-CURSO",
            "NOME-CURSO"
        ])

        instituicao = primeiro_campo(campos, [
            "NOME-INSTITUICAO",
            "NOME-DA-INSTITUICAO"
        ])

        titulo = primeiro_campo(campos, [
            "TITULO-DA-DISSERTACAO-TESE",
            "TITULO-DO-TRABALHO-DE-CONCLUSAO-DE-CURSO",
            "TITULO"
        ])

        ano_inicio = primeiro_campo(campos, [
            "ANO-DE-INICIO",
            "ANO-INICIO"
        ])

        ano_conclusao = primeiro_campo(campos, [
            "ANO-DE-CONCLUSAO",
            "ANO-CONCLUSAO"
        ])

        registro = remover_vazios({
            "tipo": tipo,
            "curso": curso,
            "instituicao": instituicao,
            "ano_inicio": ano_inicio,
            "ano_conclusao": ano_conclusao,
            "titulo_trabalho": titulo,
        })

        if registro:
            resultado.append(registro)

    return resultado


def compactar_areas_de_atuacao(conteudo):
    campos = coletar_campos(conteudo)

    areas = valores_campos(campos, [
        "NOME-GRANDE-AREA-DO-CONHECIMENTO",
        "NOME-DA-AREA-DO-CONHECIMENTO",
        "NOME-DA-SUB-AREA-DO-CONHECIMENTO",
        "NOME-DA-ESPECIALIDADE",
    ])

    return areas


def compactar_linhas_de_pesquisa(conteudo):
    if not isinstance(conteudo, list):
        return conteudo

    resultado = []

    for item in conteudo:
        campos = coletar_campos(item)

        nome = primeiro_campo(campos, [
            "NOME-DA-LINHA-DE-PESQUISA",
            "TITULO-DA-LINHA-DE-PESQUISA",
            "LINHA-DE-PESQUISA"
        ])

        objetivo = primeiro_campo(campos, [
            "OBJETIVOS-LINHA-DE-PESQUISA",
            "OBJETIVO-DA-LINHA-DE-PESQUISA",
            "DESCRICAO-DA-LINHA-DE-PESQUISA"
        ])

        registro = remover_vazios({
            "linha": nome,
            "descricao": objetivo,
        })

        if registro:
            resultado.append(registro)

    return resultado


def compactar_ensino(conteudo):
    if not isinstance(conteudo, list):
        return conteudo

    resultado = []

    for item in conteudo:
        campos = coletar_campos(item)

        disciplinas = valores_campos_contendo(
            campos,
            ["DISCIPLINA"]
        )

        curso = valores_campos(campos, [
            "NOME-DO-CURSO",
            "NOME-CURSO"
        ])

        ano_inicio = valores_campos(campos, [
            "ANO-INICIO",
            "ANO-DE-INICIO"
        ])

        ano_fim = valores_campos(campos, [
            "ANO-FIM",
            "ANO-DE-FIM"
        ])

        registro = remover_vazios({
            "curso": curso,
            "disciplinas": disciplinas,
            "ano_inicio": ano_inicio,
            "ano_fim": ano_fim,
        })

        if registro:
            resultado.append(registro)

    return resultado


def preparar_secao_curriculo(secao, conteudo):

    if secao == "producao_bibliografica":
        return preparar_producao_bibliografica(
            conteudo,
            limite=LIMITES_REGISTROS_CURRICULO["producao_bibliografica"]
        )

    if secao == "producao_tecnica":
        return preparar_producao_tecnica(
            conteudo,
            limite=LIMITES_REGISTROS_CURRICULO["producao_tecnica"]
        )

    if secao == "orientacoes_concluidas":
        return preparar_orientacoes(
            conteudo,
            limite=LIMITES_REGISTROS_CURRICULO["orientacoes_concluidas"]
        )

    if secao == "demais_trabalhos":
        return preparar_demais_trabalhos(
            conteudo,
            limite=LIMITES_REGISTROS_CURRICULO["demais_trabalhos"]
        )

    if secao == "atuacao_profissional":
        return compactar_atuacao_profissional(conteudo)

    if secao == "formacao_academica":
        return compactar_formacao_academica(conteudo)

    if secao == "areas_de_atuacao":
        return compactar_areas_de_atuacao(conteudo)

    if secao == "linhas_de_pesquisa":
        return compactar_linhas_de_pesquisa(conteudo)

    if secao == "ensino":
        return compactar_ensino(conteudo)

    return conteudo


def estruturar_curriculo(curriculo: dict) -> dict:
    """
    Mantém SOMENTE SECOES_USADAS. O id_autor não é enviado como evidência
    semântica ao modelo.

    Depois da limpeza, aplica o limite estrutural de registros nas seções de
    alta cardinalidade. Essa seleção ocorre ANTES da montagem do prompt e
    ANTES do limite final de MAX_INPUT_TOKENS.
    """
    perfil = {}

    for secao in SECOES_USADAS:
        conteudo = obter_secao_curriculo(curriculo, secao)
        if conteudo not in (None, "", [], {}):
            conteudo = limpar_estrutura_curriculo(conteudo)
            conteudo = preparar_secao_curriculo(secao, conteudo)
            if conteudo not in (None, "", [], {}):
                perfil[secao] = conteudo

    return perfil

In [ ]:
# =============================================================================
# CÉLULA 9 — Localizar os XMLs
# =============================================================================

arquivos_xml = sorted(
    arquivo
    for arquivo in os.listdir(PASTA_XML)
    if arquivo.lower().endswith(".xml")
)

print("XMLs encontrados na pasta:", len(arquivos_xml))

if not arquivos_xml:
    raise RuntimeError(f"Nenhum arquivo .xml foi encontrado em {PASTA_XML}")

print("\nPrimeiros arquivos:")
for arquivo in arquivos_xml[:5]:
    print(" -", arquivo)


In [ ]:
# =============================================================================
# CÉLULA 10 — XML -> currículo anonimizado -> currículo estruturado FINAL
# =============================================================================

perfis_estruturados = {}

xml_fora_conjunto = []
xml_com_erro = []
xml_sem_conteudo_final = []

for arquivo in tqdm(arquivos_xml, desc="Processando XMLs"):

    id_xml = normalizar_id(arquivo)

    # Mantém somente os autores pertencentes ao conjunto experimental.
    if id_xml not in autores_validos:
        xml_fora_conjunto.append(id_xml)
        continue

    caminho_xml = os.path.join(PASTA_XML, arquivo)

    # 1. Extrai e anonimiza o XML.
    curriculo_extraido = extrair_curriculo(
        caminho_xml,
        id_xml,
    )

    if curriculo_extraido is None:
        xml_com_erro.append(id_xml)
        continue

    # 2. Aplica a estrutura, seleção dos campos e limites testados.
    perfil_final = estruturar_curriculo(
        curriculo_extraido
    )

    if not perfil_final:
        xml_sem_conteudo_final.append(id_xml)
        continue

    # Mesmo padrão de ID utilizado nos experimentos.
    perfis_estruturados[id_xml] = perfil_final

print("\nProcessamento concluído.")
print("Autores esperados no qrels :", len(autores_validos))
print("XMLs existentes            :", len(arquivos_xml))
print("Perfis finais gerados      :", len(perfis_estruturados))
print("XMLs fora do conjunto      :", len(xml_fora_conjunto))
print("XMLs com erro de leitura   :", len(xml_com_erro))
print("Sem conteúdo após filtros  :", len(xml_sem_conteudo_final))


In [ ]:
# =============================================================================
# VALIDAÇÃO EXTRA — produção bibliográfica deve ter 1 registro por publicação
# =============================================================================

problemas_bibliografia = []

for autor, perfil in perfis_estruturados.items():
    producoes = perfil.get("producao_bibliografica", [])

    if len(producoes) > LIMITES_REGISTROS_CURRICULO["producao_bibliografica"]:
        problemas_bibliografia.append(
            (autor, "acima_do_limite", len(producoes))
        )

    for i, registro in enumerate(producoes):
        if not isinstance(registro, dict):
            problemas_bibliografia.append(
                (autor, f"registro_{i}", "nao_dict")
            )
            continue

        # Um registro útil precisa conter ao menos uma evidência além do tipo.
        if not any(
            campo in registro
            for campo in ("titulo", "ano", "natureza", "palavras_chave", "areas")
        ):
            problemas_bibliografia.append(
                (autor, f"registro_{i}", "sem_evidencia")
            )

print("Problemas na produção bibliográfica:", len(problemas_bibliografia))

if problemas_bibliografia:
    for item in problemas_bibliografia[:20]:
        print(item)

assert not problemas_bibliografia, (
    "Há problemas na produção bibliográfica estruturada."
)


In [ ]:
# =============================================================================
# CÉLULA 11 — Salvar diretamente o JSON final
# =============================================================================

os.makedirs(
    os.path.dirname(ARQUIVO_SAIDA),
    exist_ok=True,
)

with open(
    ARQUIVO_SAIDA,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        perfis_estruturados,
        f,
        ensure_ascii=False,
        indent=2,
    )

tamanho_mb = os.path.getsize(ARQUIVO_SAIDA) / (1024 ** 2)

print("Arquivo salvo com sucesso.")
print("Caminho :", ARQUIVO_SAIDA)
print("Autores :", len(perfis_estruturados))
print(f"Tamanho : {tamanho_mb:.2f} MB")


In [ ]:
# =============================================================================
# CÉLULA 12 — Verificar se os autores esperados foram encontrados
# =============================================================================

ids_processados = set(perfis_estruturados.keys())

ausentes = {
    normalizar_id(a)
    for a in autores_validos
} - ids_processados

print("Autores do conjunto sem perfil final:", len(ausentes))

if ausentes:
    print("\nPrimeiros IDs ausentes:")
    for autor in sorted(ausentes)[:30]:
        print(" - ID_" + autor)


In [ ]:
# =============================================================================
# CÉLULA 13 — Validar se somente as seções permitidas foram gravadas
# =============================================================================

secoes_invalidas = {}

for autor, perfil in perfis_estruturados.items():
    extras = set(perfil.keys()) - set(SECOES_USADAS)

    if extras:
        secoes_invalidas[autor] = sorted(extras)

print("Autores com seções fora do padrão:", len(secoes_invalidas))

if secoes_invalidas:
    for autor, extras in list(secoes_invalidas.items())[:10]:
        print(autor, extras)

assert not secoes_invalidas, (
    "Foram encontradas seções não previstas em SECOES_USADAS."
)

print(">> Validação das seções concluída.")


In [ ]:
# =============================================================================
# CÉLULA 14 — Validar os limites estruturais
# =============================================================================

violacoes = []

for autor, perfil in perfis_estruturados.items():

    for secao, limite in LIMITES_REGISTROS_CURRICULO.items():

        conteudo = perfil.get(secao)

        if isinstance(conteudo, list) and len(conteudo) > limite:

            violacoes.append(
                {
                    "autor": autor,
                    "secao": secao,
                    "quantidade": len(conteudo),
                    "limite": limite,
                }
            )

print("Violações de limite:", len(violacoes))

if violacoes:
    for item in violacoes[:20]:
        print(item)

assert not violacoes, (
    "Há seções com quantidade de registros acima dos limites definidos."
)

print(">> Todos os limites foram respeitados.")


In [ ]:
# =============================================================================
# CÉLULA 15 — Resumo por seção
# =============================================================================

contagem_autores_secao = Counter()
tamanhos_secao = defaultdict(list)

for autor, perfil in perfis_estruturados.items():

    for secao, conteudo in perfil.items():

        contagem_autores_secao[secao] += 1

        if isinstance(conteudo, (list, dict)):
            tamanhos_secao[secao].append(len(conteudo))
        else:
            tamanhos_secao[secao].append(1)

print("=" * 105)
print("RESUMO DO JSON FINAL")
print("=" * 105)

for secao in SECOES_USADAS:

    valores = tamanhos_secao.get(secao, [])

    if valores:
        print(
            f"{secao:30s} | "
            f"autores={contagem_autores_secao[secao]:4d} | "
            f"média={sum(valores)/len(valores):7.2f} | "
            f"máximo={max(valores):4d}"
        )
    else:
        print(
            f"{secao:30s} | "
            "autores=   0 | sem registros"
        )


In [ ]:
# =============================================================================
# CÉLULA 16 — Visualizar um autor do JSON final
# =============================================================================

if perfis_estruturados:

    autor_exemplo = next(iter(perfis_estruturados))

    print("Autor:", f"ID_{autor_exemplo}")

    exemplo = {
        autor_exemplo: perfis_estruturados[autor_exemplo]
    }

    texto = json.dumps(
        exemplo,
        ensure_ascii=False,
        indent=2,
    )
    print(texto[:20000])

    if len(texto) > 20000:
        print("\n[Exibição truncada. O arquivo JSON salvo está completo.]")
else:
    print("Nenhum perfil estruturado foi gerado.")


In [ ]:
# =============================================================================
# CÉLULA 17 — Reabrir o arquivo salvo e conferir integridade
# =============================================================================

with open(
    ARQUIVO_SAIDA,
    "r",
    encoding="utf-8",
) as f:
    dados_reabertos = json.load(f)

assert isinstance(dados_reabertos, dict)

assert len(dados_reabertos) == len(perfis_estruturados)

assert set(dados_reabertos.keys()) == set(
    perfis_estruturados.keys()
)

print("JSON reaberto sem erro.")
print("Autores confirmados:", len(dados_reabertos))
print("Arquivo pronto para uso nos diferentes modelos.")


# Geração das unidades curriculares para Coverage

A etapa abaixo utiliza o `curriculos_estruturados_padrao.json` recém-gerado e produz uma representação por unidades curriculares, mantendo campos textuais separados e sem n-grams.

Cada registro curricular válido constitui uma unidade do denominador da métrica Coverage.

In [ ]:
CAMINHO_CURRICULOS = ARQUIVO_SAIDA
CAMINHO_QRELS = ARQUIVO_AUTORES

print("Entrada:", CAMINHO_CURRICULOS)
print("Qrels:", CAMINHO_QRELS)
print("Saída Coverage:", CAMINHO_SAIDA_COVERAGE)

if not Path(CAMINHO_CURRICULOS).is_file():
    raise FileNotFoundError(f"Arquivo não encontrado: {CAMINHO_CURRICULOS}")

if not Path(CAMINHO_QRELS).is_file():
    raise FileNotFoundError(f"Arquivo não encontrado: {CAMINHO_QRELS}")

In [ ]:
# =============================================================================
# CÉLULA 2 — Imports
# =============================================================================

import json
import re

from collections import Counter, defaultdict
from typing import Dict, List, Optional, Any
from tqdm.auto import tqdm


In [ ]:
# =============================================================================
# CÉLULA 3 — Configuração das seções curriculares
# =============================================================================

SECOES_USADAS = [
    "formacao_academica",
    "atuacao_profissional",
    "areas_de_atuacao",
    "linhas_de_pesquisa",
    "ensino",
    "producao_bibliografica",
    "producao_tecnica",
    "orientacoes_concluidas",
    "demais_trabalhos",
]

# Campos considerados evidência textual em cada seção.
#
# IMPORTANTE:
# - "ano", "ano_inicio" e "ano_conclusao" não entram: são metadados temporais.
# - os campos não são concatenados;
# - listas permanecem como pedaços individuais;
# - não há n-grams.
CAMPOS_TEXTUAIS_POR_SECAO = {
    "formacao_academica": [
        "curso",
        "titulo_trabalho",
    ],

    "atuacao_profissional": [
        "instituicao",
        "vinculo_cargo",
    ],

    "linhas_de_pesquisa": [
        "linha",
        "descricao",
    ],

    "ensino": [
        "curso",
        "disciplinas",
    ],

    "producao_bibliografica": [
        "titulo",
        "natureza",
        "palavras_chave",
        "areas",
    ],

    "producao_tecnica": [
        "titulo",
        "natureza",
        "palavras_chave",
        "areas",
    ],

    "orientacoes_concluidas": [
        "titulo",
        "natureza",
        "palavras_chave",
        "areas",
    ],

    "demais_trabalhos": [
        "titulo",
        "natureza",
        "palavras_chave",
        "areas",
    ],
}

print("Seções consideradas no Coverage:", len(SECOES_USADAS))


In [ ]:
# =============================================================================
# CÉLULA 4 — Limpeza leve idêntica à lógica usada no pipeline
# =============================================================================

RE_HTML = re.compile(r"<[^>]+>")
RE_ENTIDADE_HTML = re.compile(r"&[a-z]+;|&#\d+;")
RE_CARACTERES_RUIM = re.compile(
    r"[^\w\s.,;:!?()\-'\"áéíóúâêîôûãõàèìòùçÁÉÍÓÚÂÊÎÔÛÃÕÀÈÌÒÙÇñÑüÜ]",
    flags=re.UNICODE,
)
RE_ESPACOS = re.compile(r"\s+")


def limpar_texto_basico(texto: str) -> str:
    """
    Limpeza leve:
      - remove HTML e entidades;
      - remove caracteres exóticos;
      - colapsa whitespace;
      - preserva case e acentos.
    """
    if not texto:
        return ""

    texto = RE_HTML.sub(" ", str(texto))
    texto = RE_ENTIDADE_HTML.sub(" ", texto)
    texto = RE_CARACTERES_RUIM.sub(" ", texto)
    texto = RE_ESPACOS.sub(" ", texto).strip()

    return texto


In [ ]:
# =============================================================================
# CÉLULA 5 — Normalização de IDs e carregamento do qrels
# =============================================================================

def normalizar_id_autor(autor: str) -> str:
    autor = str(autor).strip()

    if autor.upper().startswith("ID_"):
        autor = autor[3:]

    return autor


def carregar_mapa_autores_qrels(caminho: str) -> Dict[str, str]:
    """
    Retorna:
        id_normalizado -> chave_exata_do_qrels

    Assim o arquivo de Coverage usa exatamente a mesma chave que o restante
    do pipeline de avaliação.
    """
    mapa = {}

    with open(caminho, "r", encoding="utf-8") as f:
        for linha in f:
            partes = linha.rstrip("\n").split("\t")

            if not partes or not partes[0].strip():
                continue

            autor_original = partes[0].strip()
            autor_norm = normalizar_id_autor(autor_original)

            if autor_norm not in mapa:
                mapa[autor_norm] = autor_original

    return mapa


mapa_qrels = carregar_mapa_autores_qrels(CAMINHO_QRELS)

print("Autores únicos no qrels:", len(mapa_qrels))


In [ ]:
# =============================================================================
# CÉLULA 6 — Utilitários para transformar campos em pedaços naturais
# =============================================================================

def adicionar_textos(valor: Any, destino: List[str]) -> None:
    """
    Extrai strings de um campo SEM concatená-las.

    Aceita:
      - string
      - lista
      - dicionário

    Apenas strings não vazias são adicionadas.
    """
    if valor is None:
        return

    if isinstance(valor, str):
        texto = limpar_texto_basico(valor)

        if texto:
            destino.append(texto)

        return

    if isinstance(valor, list):
        for item in valor:
            adicionar_textos(item, destino)
        return

    if isinstance(valor, dict):
        for subvalor in valor.values():
            adicionar_textos(subvalor, destino)
        return


def deduplicar_preservando_ordem(textos: List[str]) -> List[str]:
    vistos = set()
    saida = []

    for texto in textos:
        if texto not in vistos:
            vistos.add(texto)
            saida.append(texto)

    return saida


def extrair_campos_selecionados(
    registro: dict,
    campos: List[str],
) -> List[str]:
    pedacos = []

    for campo in campos:
        if campo in registro:
            adicionar_textos(
                registro[campo],
                pedacos,
            )

    return deduplicar_preservando_ordem(pedacos)


In [ ]:
# =============================================================================
# CÉLULA 7 — Conversão de UMA seção em unidades curriculares
# =============================================================================

def unidades_de_secao(
    secao: str,
    conteudo: Any,
) -> Dict[str, List[str]]:
    """
    Converte uma seção do currículo em unidades independentes.

    Cada unidade resultante corresponde a UM registro curricular.
    """
    unidades = {}

    # -------------------------------------------------------------------------
    # ÁREAS DE ATUAÇÃO
    # Cada área já é uma evidência textual independente no JSON padronizado.
    # -------------------------------------------------------------------------
    if secao == "areas_de_atuacao":

        if not isinstance(conteudo, list):
            conteudo = [conteudo]

        indice = 0

        for area in conteudo:
            pedacos = []
            adicionar_textos(area, pedacos)
            pedacos = deduplicar_preservando_ordem(pedacos)

            if not pedacos:
                continue

            doc_id = f"{secao}_{indice:03d}"
            unidades[doc_id] = pedacos
            indice += 1

        return unidades

    # -------------------------------------------------------------------------
    # DEMAIS SEÇÕES
    # Cada dicionário da lista corresponde a uma unidade curricular.
    # -------------------------------------------------------------------------
    if not isinstance(conteudo, list):
        conteudo = [conteudo]

    campos = CAMPOS_TEXTUAIS_POR_SECAO.get(secao,[])
    indice = 0

    for registro in conteudo:

        if not isinstance(registro, dict):
            pedacos = []
            adicionar_textos(registro, pedacos)
            pedacos = deduplicar_preservando_ordem(pedacos)
        else:
            pedacos = extrair_campos_selecionados(registro,campos)
        if not pedacos:
            continue

        doc_id = f"{secao}_{indice:03d}"
        unidades[doc_id] = pedacos
        indice += 1

    return unidades


In [ ]:
# =============================================================================
# CÉLULA 8 — Converter UM currículo completo
# =============================================================================

def construir_unidades_curriculares(
    curriculo: dict,
) -> Dict[str, List[str]]:
    """
    Produz:
        {
            "formacao_academica_000": [...],
            "atuacao_profissional_000": [...],
            "areas_de_atuacao_000": [...],
            ...
        }
    """
    unidades = {}

    for secao in SECOES_USADAS:

        conteudo = curriculo.get(secao)

        if conteudo in (None, "", [], {}):
            continue

        unidades_secao = unidades_de_secao(secao, conteudo)
        unidades.update(unidades_secao)

    return unidades


In [ ]:
# =============================================================================
# CÉLULA 9 — Carregar o JSON curricular padronizado
# =============================================================================

with open(CAMINHO_CURRICULOS,"r",encoding="utf-8") as f:
    curriculos = json.load(f)

assert isinstance(curriculos, dict)

print("Currículos carregados:", len(curriculos))


In [ ]:
# =============================================================================
# CÉLULA 10 — Gerar perfis_documento_curriculo.json
# =============================================================================

perfis_documento_curriculo: Dict[
    str,
    Dict[str, List[str]]
] = {}

autores_fora_qrels = []
autores_sem_unidades = []

for chave_autor, curriculo in tqdm(
    curriculos.items(),
    desc="Gerando unidades curriculares",
):

    autor_norm = normalizar_id_autor(chave_autor)

    # Garante exatamente a chave utilizada no ground truth.
    autor_qrels = mapa_qrels.get(autor_norm)

    if autor_qrels is None:
        autores_fora_qrels.append(chave_autor)
        continue

    if not isinstance(curriculo, dict):
        autores_sem_unidades.append(autor_qrels)
        continue

    unidades = construir_unidades_curriculares(curriculo)

    if not unidades:
        autores_sem_unidades.append(autor_qrels)
        continue

    perfis_documento_curriculo[autor_qrels] = unidades

print("\nGeração concluída.")
print("Autores com unidades curriculares:", len(perfis_documento_curriculo))
print("Autores do JSON fora do qrels:",len(autores_fora_qrels))
print("Autores sem unidade textual válida:",len(autores_sem_unidades))


In [ ]:
# =============================================================================
# VALIDAÇÃO EXTRA — alinhamento entre currículo, qrels e Coverage
# =============================================================================

autores_curriculo_norm = {
    normalizar_id_autor(a)
    for a in curriculos.keys()
}

autores_qrels_norm = set(mapa_qrels.keys())

autores_coverage_norm = {
    normalizar_id_autor(a)
    for a in perfis_documento_curriculo.keys()
}

curriculos_com_qrels = autores_curriculo_norm & autores_qrels_norm

faltando_no_coverage = (curriculos_com_qrels - autores_coverage_norm)

extras_no_coverage = (autores_coverage_norm - curriculos_com_qrels)

print("Currículos com qrels          :", len(curriculos_com_qrels))
print("Autores no Coverage           :", len(autores_coverage_norm))
print("Faltando no Coverage          :", len(faltando_no_coverage))
print("Extras no Coverage            :", len(extras_no_coverage))

if faltando_no_coverage:
    print(
        "\nAutores sem unidade textual válida "
        "(primeiros 20):",
        sorted(faltando_no_coverage)[:20],
    )

assert not extras_no_coverage, (
    "Há autores no Coverage que não pertencem simultaneamente "
    "ao JSON curricular e ao qrels."
)



In [ ]:
# =============================================================================
# CÉLULA 11 — Salvar o arquivo que será usado no Coverage
# =============================================================================

os.makedirs(os.path.dirname(CAMINHO_SAIDA_COVERAGE), exist_ok=True)

with open(CAMINHO_SAIDA_COVERAGE,"w",encoding="utf-8") as f:
    json.dump(perfis_documento_curriculo,f,ensure_ascii=False,indent=2)

tamanho_mb = (os.path.getsize(CAMINHO_SAIDA_COVERAGE)/ (1024 ** 2))

print("Arquivo salvo.")
print("Caminho :", CAMINHO_SAIDA_COVERAGE)
print("Autores :", len(perfis_documento_curriculo))
print(f"Tamanho : {tamanho_mb:.2f} MB")


In [ ]:
# =============================================================================
# CÉLULA 12 — Validação do formato esperado pela função de Coverage
# =============================================================================

erros = []

for autor, unidades in perfis_documento_curriculo.items():

    if not isinstance(unidades, dict):
        erros.append(f"{autor}: unidades não são dict")
        continue

    for doc_id, pedacos in unidades.items():
        if not isinstance(doc_id, str):
            erros.append(f"{autor}: doc_id não é string")
        if not isinstance(pedacos, list):
            erros.append(f"{autor}/{doc_id}: pedaços não são list")
            continue

        if not pedacos:
            erros.append(f"{autor}/{doc_id}: unidade vazia")
        if not all(isinstance(p, str) and p.strip() for p in pedacos):
            erros.append(f"{autor}/{doc_id}: contém pedaço inválido")

print("Erros de estrutura:", len(erros))

if erros:
    for erro in erros[:30]:
        print(" -", erro)

assert not erros, ("O arquivo não está no formato esperado pelo Coverage.")

print(">> Formato validado: {autor: {doc_id: [pedaços textuais]}}")


In [ ]:
# =============================================================================
# CÉLULA 13 — Auditoria: número de unidades por seção
# =============================================================================

contagem_total_secao = Counter()
unidades_por_autor = []

for autor, unidades in perfis_documento_curriculo.items():

    unidades_por_autor.append(len(unidades))
    for doc_id in unidades:
        for secao in SECOES_USADAS:
            prefixo = secao + "_"
            if doc_id.startswith(prefixo):
                contagem_total_secao[secao] += 1
                break

print("=" * 95)
print("UNIDADES CURRICULARES GERADAS")
print("=" * 95)

for secao in SECOES_USADAS:
    print(
        f"{secao:30s} | "
        f"{contagem_total_secao[secao]:7d}"
    )

print("-" * 95)

if unidades_por_autor:
    print("Autores:",len(unidades_por_autor))
    print("Média de unidades/autor:",round(sum(unidades_por_autor)/ len(unidades_por_autor),2))
    print("Mínimo:", min(unidades_por_autor))
    print("Máximo:",max(unidades_por_autor))


In [ ]:
# =============================================================================
# CÉLULA 14 — Mostrar um exemplo do arquivo final
# =============================================================================

if perfis_documento_curriculo:

    autor_exemplo = next(iter(perfis_documento_curriculo))

    exemplo = {
        autor_exemplo:
            perfis_documento_curriculo[
                autor_exemplo
            ]
    }

    texto = json.dumps(
        exemplo,
        ensure_ascii=False,
        indent=2,
    )

    print("Autor de exemplo:", autor_exemplo)
    print(texto[:20000])

    if len(texto) > 20000:
        print(
            "\n[Exibição truncada apenas nesta célula. "
            "O arquivo salvo está completo.]"
        )


In [ ]:
# =============================================================================
# CÉLULA 15 — Salvar relatório de auditoria
# =============================================================================

relatorio = {
    "arquivo_entrada": CAMINHO_CURRICULOS,
    "arquivo_saida_coverage": CAMINHO_SAIDA_COVERAGE,

    "definicao_coverage": (
        "unidades_curriculares_cobertas / "
        "total_unidades_curriculares"
    ),

    "autores_entrada": len(curriculos),
    "autores_saida": len(perfis_documento_curriculo),

    "autores_fora_qrels": len(autores_fora_qrels),

    "autores_sem_unidades": len(autores_sem_unidades),

    "secoes": SECOES_USADAS,

    "campos_textuais_por_secao":
        CAMPOS_TEXTUAIS_POR_SECAO,

    "total_unidades_por_secao": {
        secao: int(
            contagem_total_secao[secao]
        )
        for secao in SECOES_USADAS
    },

    "media_unidades_por_autor": (
        sum(unidades_por_autor)
        / len(unidades_por_autor)
        if unidades_por_autor
        else 0.0
    ),

    "min_unidades_por_autor": (
        min(unidades_por_autor)
        if unidades_por_autor
        else 0
    ),

    "max_unidades_por_autor": (
        max(unidades_por_autor)
        if unidades_por_autor
        else 0
    ),

    "observacoes": [
        "Não são usados n-grams.",
        "Campos textuais não são concatenados.",
        "Pedaços textuais duplicados dentro de uma unidade são removidos.",
        "Anos não são utilizados como evidência textual de Coverage.",
        "Cada registro curricular corresponde a uma unidade do denominador.",
    ],
}

with open(CAMINHO_RELATORIO,"w",encoding="utf-8") as f:
    json.dump(relatorio,f, ensure_ascii=False,indent=2)

print("Relatório salvo em:")
print(CAMINHO_RELATORIO)


In [ ]:
# =============================================================================
# CÉLULA 16 — Reabrir o arquivo final e confirmar integridade
# =============================================================================

with open(CAMINHO_SAIDA_COVERAGE,"r",encoding="utf-8",) as f:
    coverage_reaberto = json.load(f)

assert (set(coverage_reaberto.keys()) == set(perfis_documento_curriculo.keys()))

assert all(isinstance(unidades, dict)for unidades in coverage_reaberto.values())

print("JSON reaberto sem erro.")
print("Autores confirmados:", len(coverage_reaberto))

